# M2 — Watermarks: scenario testing

M2-only isolation: every verdict in this notebook is caused exclusively by
watermark evidence. Provenance, forensics, and the detector are silenced.

**Five decoders, ranked by ecosystem reach:**

| # | scheme | detects | type | verdict tier |
|---|---|---|---|---|
| 1 | dwtDct | SD 1.x/2.x/SDXL | payload match | verified |
| 2 | trustmark | Adobe Firefly | payload decode | verified |
| 3 | stable-signature-bzh | SDXL-turbo IMATAG | zero-bit (p-value) | likely |
| 4 | synthid-cnn | Google/OpenAI | learned surrogate | likely |
| 5 | meta-invisible | Meta AI | none exists | documented |

Convention: commit WITH outputs — this is a run record.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .

# Optional decoder dependencies — install what you want to test:
%pip install -q trustmark 2>/dev/null || echo "trustmark not available"
# transformers is heavy; skip if not testing Stable Signature BZH:
# %pip install -q transformers
# torch/torchvision should already be present in Colab GPU runtimes

import numpy as np
from pathlib import Path
from PIL import Image
from ai_image_id.ingest import ingest
from ai_image_id.watermark import analyze_watermarks, dwt_dct, SDXL_BITS, SD_V1_BITS
from ai_image_id.fusion import fuse
from ai_image_id.schema import Evidence, ProvenanceEvidence, SpectrumEvidence, Verdict

def report(path, label=""):
    """M2-only evidence card. No provenance, no FFT, no detector."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    evidence = Evidence(
        provenance=ProvenanceEvidence(),          # M1 silenced
        watermarks=wms,
        spectrum=SpectrumEvidence(valid=False),    # M3 silenced
        detector=None,                            # M4 silenced
    )
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    print(f"┌─ {label or path}")
    print(f"│ verdict: {r.ai_verdict.value} ({r.confidence})")
    for w in wms:
        status = "DETECTED" if w.detected else ("n/a" if not w.applicable else "not detected")
        detail = ""
        if w.bit_accuracy is not None: detail = f" acc/p={w.bit_accuracy}"
        if w.matched_payload: detail += f" payload={w.matched_payload}"
        print(f"│   {w.scheme:<22} {status}{detail}")
        if w.notes and not w.applicable: print(f"│     └ {w.notes}")
    print(f"└ notes: {r.notes}\n")
    return r

def wm_row(path):
    """One-line summary for matrix tables."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    evidence = Evidence(provenance=ProvenanceEvidence(), watermarks=wms,
                        spectrum=SpectrumEvidence(valid=False), detector=None)
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    cols = {}
    for w in wms:
        if w.scheme in ("meta-invisible",): continue  # skip documented-only
        cols[w.scheme] = w.detected if w.applicable else None
    return cols, r.ai_verdict.value

# Check which decoders are actually available in this session
print("decoder availability:")
rgb_test = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)
for w in analyze_watermarks(rgb_test):
    status = "✓ ready" if w.applicable else f"✗ {w.notes}"
    print(f"  {w.scheme:<22} {status}")

## 1 — DWT-DCT: synthetic round-trip (controlled ground truth)

Tests our vendored codec's self-consistency. The interesting question is
the fragility cliff: at what JPEG quality does the watermark die?

In [ ]:
FIX = Path("/content/fixtures_m2"); FIX.mkdir(exist_ok=True)

rng = np.random.default_rng(7)
y, x = np.mgrid[0:512, 0:512]
base = np.clip(np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83),
                         90+40*np.sin((x+y)/71)], -1) + rng.normal(0,6,(512,512,3)),
               0, 255).astype(np.uint8)

# Embed SDXL payload, save at various qualities
marked = dwt_dct.embed(base, SDXL_BITS)
for q_label, save_fn in [
    ("lossless", lambda p: Image.fromarray(marked).save(p)),
    ("q92",      lambda p: Image.fromarray(marked).save(p, quality=92)),
    ("q80",      lambda p: Image.fromarray(marked).save(p, quality=80)),
    ("q70",      lambda p: Image.fromarray(marked).save(p, quality=70)),
]:
    p = FIX/f"sdxl_{q_label}.{'png' if q_label=='lossless' else 'jpg'}"
    save_fn(p)
    report(str(p), f"SDXL payload, {q_label}")

# Clean baseline
p = FIX/"clean.png"; Image.fromarray(base).save(p)
report(str(p), "no watermark (baseline)")

## 2 — DWT-DCT fragility curve

Sweep JPEG quality 50–98: bit accuracy vs. compression. The cliff location
tells us where M2's DWT-DCT decoder stops being useful in practice.

In [ ]:
import matplotlib.pyplot as plt

qualities = list(range(50, 100, 2))
accuracies = []
for q in qualities:
    p = FIX/f"sweep_q{q}.jpg"
    Image.fromarray(marked).save(p, quality=q)
    img = ingest(p)
    wms = analyze_watermarks(img.rgb)
    best = next((w for w in wms if w.scheme == "dwtDct" and w.bit_accuracy is not None), None)
    accuracies.append(best.bit_accuracy if best else 0.5)

plt.figure(figsize=(10, 4))
plt.plot(qualities, accuracies, "o-", markersize=4)
plt.axhline(0.90, color="red", linestyle="--", label="detection threshold (0.90)")
plt.axhline(0.50, color="gray", linestyle=":", label="coin flip (0.50)")
plt.xlabel("JPEG quality"); plt.ylabel("bit accuracy")
plt.title("DWT-DCT watermark survival vs. JPEG quality")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("/content/m2_dwtdct_curve.png", dpi=150); plt.show()

cliff = next((q for q, a in zip(qualities, accuracies) if a and a >= 0.90), None)
print(f"DWT-DCT detectable (≥0.90) above Q={cliff}" if cliff else "never reaches threshold")

## 3 — Real images: the validation gauntlet

Upload real images from different generators. Expected behavior per decoder:

| image source | dwtDct | trustmark | stable-sig | synthid-cnn |
|---|---|---|---|---|
| Real SDXL (diffusers + invisible-watermark) | ✓ detect | · | · | · |
| Adobe Firefly export | · | ✓ detect | · | · |
| OpenAI/ChatGPT PNG | · | · | · | ✓ detect |
| Google Gemini image | · | · | · | ✓ detect |
| Phone photo | · | · | · | · |
| Screenshot of any AI image | · | · | · | maybe ✓ |

The last row is the key test: SynthID CNN claims transport survival.
Upload your OpenAI PNG AND its screenshot to compare.

In [ ]:
from google.colab import files
up = files.upload()
for name in sorted(up):
    report(name)

## 4 — Transport-degradation matrix (M2 signals)

Same seven hops as notebook 03 (M1). The hypothesis: watermarks survive
transforms that kill metadata. Compare column-by-column with the M1 matrix.

Upload a watermarked image when prompted (your OpenAI PNG is the best
candidate — it carries both C2PA for M1 comparison and SynthID for M2).

In [ ]:
import shutil, subprocess

up = files.upload()
orig = Path(next(iter(up)))
HOP = Path("/content/hops_m2"); HOP.mkdir(exist_ok=True)
im = Image.open(orig).convert("RGB")

hops = {"0-original": orig}
p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
p = HOP/"2-screenshot.png"; im.save(p);               hops["2-screenshot"] = p
p = HOP/"3-messenger.jpg";  im.resize((im.width//2, im.height//2)).save(p, quality=70)
hops["3-messenger"] = p
p = HOP/("4-strip"+orig.suffix); shutil.copy(orig, p)
subprocess.run(["exiftool","-overwrite_original","-all=",str(p)], capture_output=True)
hops["4-exiftool-strip"] = p
p = HOP/"5-crop.jpg"; im.crop((50,50,im.width-50,im.height-50)).save(p, quality=92)
hops["5-crop"] = p
p = HOP/"6-pil-reencode.png"; im.save(p)
hops["6-pil-reencode"] = p

SCHEMES = ["dwtDct", "trustmark", "stable-signature-bzh", "synthid-cnn"]
header = f"{'transport':<20}" + "".join(f"{s:<22}" for s in SCHEMES) + "verdict"
print(header)
for name, path in hops.items():
    cols, verdict = wm_row(path)
    row = f"{name:<20}"
    for s in SCHEMES:
        v = cols.get(s)
        row += f"{'✓' if v else ('·' if v is False else '—'):<22}"
    row += verdict
    print(row)

### Reading the matrix — M1 vs M2 comparison

Place this side-by-side with notebook 03's M1 matrix:

**M1 (provenance):** C2PA survives only exiftool-strip; everything else dies.

**M2 (watermarks) — expected pattern:**
- *dwtDct:* fragile — dies at any JPEG recompression below its cliff Q
- *trustmark:* designed for transport survival — should survive re-save, crop,
  possibly messenger. THE complementary signal to C2PA.
- *synthid-cnn:* trained against JPEG/blur/crop/screenshot — should survive
  most hops. If it does, it's the most durable signal in the entire pipeline.

The composite picture: M1 proves origin on pristine images; M2 extends
detection through reprocessing; together they cover more scenarios than
either alone. That complementarity is the evidence-fusion thesis.

## 5 — Cross-decoder agreement test

Run all decoders on the SAME image. When multiple decoders fire on the same
file, it's strong corroboration. When they disagree, it's interesting
(e.g., SynthID CNN fires but TrustMark doesn't → image is Google/OpenAI,
not Adobe). Decoder agreement patterns are implicit generator attribution.

In [ ]:
# Re-run on all uploaded images from §3, showing per-decoder verdicts side by side
import os
uploads = [f for f in os.listdir("/content") if f.endswith((".png",".jpg",".jpeg",".webp"))
           and "fixture" not in f and "sweep" not in f and "hop" not in f]
if not uploads:
    print("no uploaded images found — run §3 first")
else:
    print(f"{'image':<40}" + "".join(f"{s:<18}" for s in SCHEMES))
    for name in sorted(uploads):
        cols, verdict = wm_row(f"/content/{name}")
        row = f"{name[:38]:<40}"
        for s in SCHEMES:
            v = cols.get(s)
            row += f"{'✓' if v else ('·' if v is False else '—'):<18}"
        print(f"{row}  → {verdict}")

## 6 — Scope: what M2 covers and doesn't

| generator | watermark scheme | M2 decoder | status |
|---|---|---|---|
| SD 1.x/2.x/SDXL (with invisible-watermark) | DWT-DCT | `_check_dwtdct` | implemented |
| Adobe Firefly | TrustMark | `_check_trustmark` | implemented |
| SDXL-turbo IMATAG builds | Stable Signature BZH | `_check_stable_signature_bzh` | implemented |
| Google Gemini / Imagen | SynthID | `_check_synthid_cnn` (surrogate) | implemented (learned, not cryptographic) |
| OpenAI GPT-Image-2 | SynthID | `_check_synthid_cnn` (surrogate) | implemented (learned, not cryptographic) |
| Meta AI | proprietary | none | **blocked** — no public decoder |
| Midjourney | none known | — | no watermark to detect |
| Flux, Leonardo, Ideogram, Recraft | none known | — | no watermark to detect |

Honest boundary: decoders 1–2 verify payloads (cryptographic confidence);
3–4 detect statistical presence (learned confidence, ~3% FP possible).
For generators with no watermark, only M1 (metadata) and M4 (classifier)
provide signal.